# 🌱 Simulation of Wheat Vegetative Growth with WheatFSPM based on CN metabolism and hydraulics

This tutorial illustrates how to use the WheatFspm model to simulate the growth and functioning of a wheat plant during the vegetative stages. Here, both the metabolic and hydraulic regulations are taken into account. This example is based on the script [main.py](https://github.com/openalea/WheatFspm/blob/master/example/Vegetative_stages_hydraulics/main.py).

The initial conditions are the same as in the first notebook except for the water-related variables that have been added. Plant organs and compartments are similar except that a xylem is added at the axis scale.

## 🛠️ Configuration and prerequisites
Let's import all the required packages

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Adel-Wheat dependencies (https://github.com/openalea/adel)
from openalea.adel.adel_dynamic import AdelDyn
from openalea.adel.echap_leaf import echap_leaves

from openalea.widgets.plantgl import PlantGL

## Model inputs
* Weather data

   They should contain the following variables :
    * *t*: the time index used in the simulation (hour)
    * *DOY*: the Day Of the Year
    * *hour*: the hour of the day [1-24]
    * *air_temperature*: temperature of the air (°C)
    * *soil_temperature*: temperature of the soil (°C)
    * *PARi*: incident Photosynthetically Active Radiation above the canopy (µmol m-2 s-1)
    * *humidity*: air humidity (relative)
    * *ambient_CO2*: CO2 concentration of the atmosphere (ppm)
    * *Wind* : wind speed 2m above the canopy (m s-1)


In [ ]:
INPUTS_DIRPATH=os.path.join('inputs', 'Hydraulics')

# Loading a meteo file and display head
METEO_FILENAME = "meteo_file.csv"
meteo_df = pd.read_csv(os.path.join(INPUTS_DIRPATH, METEO_FILENAME), delimiter=',',  index_col='t')
meteo_df.head()

* Setting management:

   The management options should be set in a csv file. The file should have the following variables:
   * *plant_density*: the sowing density per genotype in plants per square meter. Provide it as a dict {genotype_id: sowing_density_value}
   * *N_fertilizations*: the fertilization regime in µmol of N). Provide it as a dict {hour of the simulation: fertilisation_value}

In [ ]:
MANAGEMENT_FILENAME = 'management.csv'
print(pd.read_csv(os.path.join(INPUTS_DIRPATH, MANAGEMENT_FILENAME)))

* Initial conditions

  The initial state of the different plant organs and the soil are loaded from different csv files. They contain the initial dimensions, masses and metabolic contents. In the *hydraulics* version, inputs should also provide some initial water related variables (e.g. water_content, turgor potential, width, thickness).

In [ ]:
# Name of the CSV files which describes the initial state of the system
AXES_INITIAL_STATE_FILENAME = 'axes_initial_state.csv'  # Axis scale
ORGANS_INITIAL_STATE_FILENAME = 'organs_initial_state.csv'  # Organ scale (contains data for the roots and phloem compartments)
HIDDENZONES_INITIAL_STATE_FILENAME = 'hiddenzones_initial_state.csv'  # Hidden zone scale
ELEMENTS_INITIAL_STATE_FILENAME = 'elements_initial_state.csv'  # Photosynthetic element scale
SOILS_INITIAL_STATE_FILENAME = 'soils_initial_state.csv'  # Soil


# Read the inputs from CSV files and create inputs dataframes

for inputs_filename in (AXES_INITIAL_STATE_FILENAME,
                        ORGANS_INITIAL_STATE_FILENAME,
                        HIDDENZONES_INITIAL_STATE_FILENAME,
                        ELEMENTS_INITIAL_STATE_FILENAME,
                        SOILS_INITIAL_STATE_FILENAME):
    inputs_dataframe = pd.read_csv(os.path.join(INPUTS_DIRPATH, inputs_filename))
    print(inputs_filename, '\n', inputs_dataframe.head())

Note the decomposition of the plant into axes, metamers, organs and elements. More information can be found in the [documentation](https://wheatfspm.readthedocs.io/en/latest/?badge=latest).

* Configuration of Adel-Wheat

  WheatFspm includes an implicit coupling to the model [Adel-Wheat](https://adel.readthedocs.io/en/latest/?badge=latest). This models allows for the 3D representation of the wheat architecture and the generation of the MTG object used for the communication (read/write) between each submodule of WheatFspm. The inputs should therefore contain a pickle of an Adel MTG (adel0000.pckl and scene0000.bgeom) and two tables to set the plant architecture in Adel (adel_pars.RData and phytoT.csv).

  In this example, we will use the leaf database for the wheat cultivar Soissons.

In [ ]:
#- Illustration of the initial MTG used the vegetative stages example (starts at leaf 4 emergence).
# Instantiates AdelDyn object with Soissons database and set scene unit to meter
adel_wheat = AdelDyn(seed=1, scene_unit='m', leaves=echap_leaves(xy_model='Soissons_byleafclass'))
# Instantiates a MTG object from a pickle
g = adel_wheat.load(directory=INPUTS_DIRPATH)

# Visualisation of the input wheat architecture
scene = adel_wheat.scene(g)
mesh=PlantGL(scene)
mesh

Exploring the MTG
* The phloem, xylem, roots and grains compartment are stored at the axis vertex of the MTG. The different metabolic, dimension and mass related-properties are stored as properties of those compartments.
* The hiddenzone compartments are stored at each metamer vertex of the MTG.
* Each metamer is  characterized by 3 MTG components : a blade, a sheath and an internode
* Each organ can be composed by 2 elements, which are used to separate the hidden and visible part of the organ

* Configuration of outputs and postprocessing.

  By default, the model will write raw outputs (one csv file per scale) in a folder *outputs* and postprocessing (one csv file per scale) in a folder *postprocessing*.

In [ ]:
# -- OUTPUT CONFIGURATION --
OUTPUTS_DIRPATH = os.path.join('outputs', 'simple_run_vegetative_stages_hydraulics')

# -- POSTPROCESSING CONFIGURATION --
POSTPROCESSING_DIRPATH= os.path.join('postprocessing', 'simple_run_vegetative_stages_hydraulics')

# -- POSTPROCESSING CONFIGURATION --
GRAPH_DIRPATH= os.path.join('graphs', 'simple_run_vegetative_stages_hydraulics')

## Run of the simulation
Running the simulation from the *main.py* by calling *openalea.fspmwheat.fspmwheat_runner* with the argument *hydraulics=True*.

In [ ]:
from openalea.fspmwheat.fspmwheat_runner import run as fspmwheat_runner

SIMULATION_LENGTH = 48  # length of the simulation (hours)
START_TIME = 0

# precision of floats used to write and format the output CSV files
OUTPUTS_PRECISION = 2

# Rates and time-dependant parameters in WheatFspm are expressed in s or s-1 but simulations (and differential equations solve) are usually integrated over an hour
HOUR_TO_SECOND_CONVERSION_FACTOR = 3600

fspmwheat_runner(simulation_length=SIMULATION_LENGTH, forced_start_time=0,
                run_simu=True, run_postprocessing=True, generate_graphs=True, run_from_outputs=False,
                tillers_replications={'T1': 0.5, 'T2': 0.5, 'T3': 0.5, 'T4': 0.5}, heterogeneous_canopy=True,
                METEO_FILENAME=METEO_FILENAME, MANAGEMENT_FILENAME=MANAGEMENT_FILENAME,
                INPUTS_DIRPATH=INPUTS_DIRPATH, OUTPUTS_DIRPATH=OUTPUTS_DIRPATH, POSTPROCESSING_DIRPATH=POSTPROCESSING_DIRPATH, GRAPHS_DIRPATH=GRAPH_DIRPATH,
                hydraulics=True)



## Generate outputs, postprocessing and graphs
Outputs relate to the different variable solved by the model, while the postprocessing include extra-calculations allowing for in-depth interpretation of the simulations. Outputs and postprocessing are written as csv files, one per scale (axis, organs, hiddenzones, elements and soil). Files are stored in /outputs/simple_run_vegetative_stages and /postprocessing/simple_run_vegetative_stages.
At the end of the simulation, main prags are automatically plotted and stored in the specified folder

## 📈 Key Results Visualization

In [ ]:
axes_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'axes_postprocessing.csv'))
organs_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'organs_postprocessing.csv'))
organs_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'organs_postprocessing.csv'))
xylem_postprocessing_df = organs_postprocessing_df[organs_postprocessing_df['organ'] == 'xylem']
hz_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'hiddenzones_postprocessing.csv'))
hz3_MS_postprocessing_df = hz_postprocessing_df[(hz_postprocessing_df['axis'] == 'MS') & (hz_postprocessing_df['metamer'] == 3)]
elements_postprocessing_df = pd.read_csv(os.path.join(POSTPROCESSING_DIRPATH, 'elements_postprocessing.csv'))
leaf3_postprocessing_df = elements_postprocessing_df[(elements_postprocessing_df['axis'] == 'MS') & (elements_postprocessing_df['metamer'] == 3) & (elements_postprocessing_df['element'] == 'LeafElement1')]


# Xylem water potential
plt.figure(figsize=(10, 6))
plt.plot(xylem_postprocessing_df['t'], xylem_postprocessing_df['water_potential'])
plt.xlabel('Time (hour)')
plt.ylabel('Xylem water potential (MPa)')
plt.grid(True)
plt.show()

# Leaf 3 water status
plt.figure(figsize=(10, 6))
plt.plot(hz3_MS_postprocessing_df['t'], hz3_MS_postprocessing_df['water_potential'], label='hz water potential', color='blue')
plt.plot(hz3_MS_postprocessing_df['t'], hz3_MS_postprocessing_df['turgor_water_potential'], label='hz turgor potential', color='green')
plt.plot(hz3_MS_postprocessing_df['t'], hz3_MS_postprocessing_df['osmotic_water_potential'], label='hz osmotic potential', color='yellow')
plt.plot(leaf3_postprocessing_df['t'], leaf3_postprocessing_df['water_potential'], label='blade water potential', color='blue', linestyle='--')
plt.plot(leaf3_postprocessing_df['t'], leaf3_postprocessing_df['turgor_water_potential'], label='blade turgor potential', color='green', linestyle='--')
plt.plot(leaf3_postprocessing_df['t'], leaf3_postprocessing_df['osmotic_water_potential'], label='blade osmotic potential', color='yellow', linestyle='--')

plt.xlabel('Time (hour)')
plt.ylabel('Leaf 3 Potential (MPa)')
plt.legend()
plt.grid(True)
plt.show()